| 类别 | 方法/装饰器 | 一句话描述 |
| :--- | :--- | :--- |
| 基础 | `@ray.remote` | 把函数/类变成分布式任务/Actor |
| | `ray.get()` | 阻塞：拿结果 |
| | `ray.put()` | 存数：存大数据 |
| 流程控制 | `ray.wait()` | 异步：谁先做完先处理谁 (实现动态调度) |
| | `.remote()` | 执行：触发远程调用 |
| 状态管理 | `ray.get_actor()` | 寻址：通过名字找到已存在的 Actor |
| | `ray.kill()` | 销毁：强制杀掉 Actor |
| 调试/运维 | `ray.nodes()` | 查看集群节点状态 |
| | `ray.cluster_resources()` | 查看集群总资源 |
| | `ray.timeline()` | 生成性能分析轨迹文件 |



# ray方法集合
如果把 Ray 比作一个操作系统，remote 是创建进程，put 是写文件，get 是读文件，那么下面这些方法就是用来管理“进程调度”、“异步等待”和“系统状态”的高级指令。
以下是 Ray 中其他最常用的核心方法：
## 异步与流程控制 (核心)

这是 Ray 区别于普通 Python 脚本的关键，让你能写出非阻塞的并发代码。

- <font color='red'>ray.wait(object_refs, num_returns=1, timeout=None)</font>
    - <font color='red'>学会 ray.wait 是脱离“新手村”的标志，它能让你写出吞吐量更高、响应更快的分布式程序</font>
    - 作用：“谁先做完先处理谁”。它允许你等待一批任务中的任意一个（或多个）完成，而不是像 ray.get 那样傻等所有任务结束。
    - 返回值：返回两个列表，一个是已完成的任务引用列表，一个是未完成的列表。
    - 场景：
        - 动态负载均衡：你有 100 个任务，但只有 10 个 Worker。你可以提交一批任务，用 ray.wait 监听谁先做完，然后立刻把新任务发给那个空闲的 Worker。
        - 流式处理：就像视频播放，不需要下载完整个电影（所有任务）才开始看，而是下载好一部分（ray.wait 返回）就开始处理。
- <font color='red'>ray.get_actor(name, namespace=None)</font>
    - 作用：“寻找失散的 Actor”。通常 Actor 是通过变量引用的，但如果 Driver 程序重启了，变量丢了怎么办？你可以给 Actor 起个名字（name="MyCounter"），然后用这个方法重新获取它的引用。
    - 场景：构建有状态的服务（如全局计数器、参数服务器），即使客户端断开重连，也能连上同一个服务实例。
    
## 集群与生命周期管理

用于控制 Ray 集群本身的运行状态。

- <font color='red'>ray.init(...)</font>
    - 作用：“启动引擎”。初始化 Ray 环境。
    - 细节：虽然它是启动方法，但它包含很多参数，比如 address（连接现有集群）、num_cpus（限制本地使用的核数）、log_to_driver（是否打印日志）等。
- <font color='red'>ray.shutdown()</font>
    - 作用：“关闭引擎”。断开与集群的连接并清理资源。通常在脚本结束时自动调用，但在交互式开发（Jupyter Notebook）中，为了重置环境，经常手动调用。
- <font color='red'>ray.kill(actor)</font>
    - 作用：“强制销毁”。强制终止一个 Actor。
    - 场景：当 Actor 陷入死循环、内存泄漏或者你需要释放资源给其他任务时。
    
## 调试与状态查询 (可观测性)

当你需要知道“现在集群里到底在发生什么”时，这些方法非常有用。

- <font color='red'>ray.nodes()</font>
    - 作用：“点名”。返回集群中所有节点的列表（包含 Head 和 Worker 节点）。
    - 用途：查看哪些节点是 Alive 的，哪些挂了，或者查看每个节点的 IP 和 ID。
- <font color='red'>ray.cluster_resources()</font>
    - 作用：“盘点家底”。返回整个集群可用的资源总量（CPU、GPU、内存等）。
    - 用途：在提交任务前，先看看资源够不够，或者根据资源总量动态调整任务数量。
- <font color='red'>ray.timeline(filename)</font>
    - 作用：“录屏”。生成一个 Chrome Trace 格式的文件。
    - 用途：性能分析的神器。你可以把这个文件加载到 Chrome 浏览器的 chrome://tracing 中，可视化地看到每个任务在哪个时间点、哪个节点上执行，以及等待了多久。
    
## 高级配置 (Runtime Env)
- <font color='red'>ray.init(runtime_env={...}) 或 @ray.remote(runtime_env={...})</font>
    - 作用：“自带环境”。
    - 场景：默认情况下，Ray 假设所有节点的环境（Python 包、文件）都是一样的。如果你的任务依赖特殊的 requirements.txt 或者特定的本地文件，你可以通过 runtime_env 告诉 Ray：“执行这个任务前，先帮我把这些 pip 包装好，或者把那个文件传过来”。这解决了依赖地狱问题。